# NLP Project — Data Generation

This notebook builds the two datasets used by the project:

1. **ICL pattern-discovery stream** — synthetic classification prompts with arbitrary labels.
2. **AGENDA semantic-RI stream** — relation examples built from AGENDA after replacing entities with single capital letters.

The notebook is organized so that configuration, reusable helpers, generation, validation, and saving are separated clearly.

### Output files

All generated files are written to `DATA_DIR`:

- `icl_stream.jsonl`
- `ri_agenda_stream.jsonl`
- `ri_agenda_olmo_safe.jsonl`

> Run the notebook from top to bottom. The tokenizer loaded in Section 2 is reused by the AGENDA token-safety audit.

## 1. Setup

Install dependencies, mount Google Drive, define project paths, and import shared libraries.

In [ ]:
!pip -q install transformers huggingface_hub pandas spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
import random
import re
import string
from pathlib import Path

import pandas as pd
import spacy
from transformers import AutoTokenizer

In [ ]:
# Project paths
ROOT = Path("/content/drive/MyDrive/NLP_Project/olmo_sih_dynamics")
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Model used for tokenizer-safety checks
MODEL_NAME = "allenai/OLMo-2-1124-7B"

# Shared reproducibility settings
BASE_SEED = 20260923

print("Data directory:", DATA_DIR)
print("Tokenizer model:", MODEL_NAME)

In [ ]:
def write_jsonl(rows, path):
    """Write a list of dictionaries to a JSONL file."""
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")
    print(f"Saved {len(rows):,} rows -> {path}")

# 2. Data Stream 1 — ICL Pattern Discovery

The synthetic ICL task uses arbitrary numeric labels. The model therefore cannot solve the task from world knowledge alone; it must infer the label mapping from the demonstrations.

The four task families are:

- **Binary fruit/month**
- **Binary furniture/profession**
- **Four-class fruit/month**
- **Nine-class fruit/animal/month**

For the binary case with categories For two categories \(E_1,E_2\):

$$ (E_1,E_2)\rightarrow0 $$ $$ (E_2,E_1)\rightarrow1. $$

The lexical inventories below are a controlled implementation choice because the paper does not publish its complete word lists. They are kept fixed across checkpoints.

### 2.1 Category vocabularies and task definitions

In [ ]:
CATEGORIES = {
    "fruit": [
        "apple", "orange", "banana", "pear", "grape",
        "peach", "plum", "mango", "lemon", "cherry",
        "apricot", "melon", "kiwi", "papaya", "lime",
        "coconut", "fig", "guava", "nectarine", "pomegranate",
    ],
    "month": [
        "January", "February", "March", "April",
        "May", "June", "July", "August",
        "September", "October", "November", "December",
    ],
    "furniture": [
        "chair", "table", "sofa", "desk", "bed",
        "stool", "shelf", "cabinet", "dresser", "bench",
        "wardrobe", "bookcase", "couch", "armchair",
        "nightstand", "cupboard",
    ],
    "profession": [
        "doctor", "teacher", "lawyer", "nurse", "engineer",
        "chef", "farmer", "pilot", "artist", "dentist",
        "plumber", "carpenter", "architect", "mechanic",
        "scientist", "accountant",
    ],
    "animal": [
        "dog", "cat", "horse", "tiger", "lion",
        "zebra", "rabbit", "monkey", "panda", "otter",
        "fox", "bear", "wolf", "giraffe", "leopard",
        "kangaroo",
    ],
}

In [ ]:
TASKS = {
    "binary_fruit_month": {
        "categories": ["fruit", "month"],
        "mapping": {
            (0, 1): "0",
            (1, 0): "1",
        },
    },
    "binary_furniture_profession": {
        "categories": ["furniture", "profession"],
        "mapping": {
            (0, 1): "0",
            (1, 0): "1",
        },
    },
    "four_class": {
        "categories": ["fruit", "month"],
        "mapping": {
            (0, 0): "0",
            (0, 1): "1",
            (1, 0): "2",
            (1, 1): "3",
        },
    },
    "nine_class": {
        "categories": ["fruit", "animal", "month"],
        "mapping": {
            (i, j): str(3 * i + j)
            for i in range(3)
            for j in range(3)
        },
    },
}

# For the nine-class task: label = 3*i + j.

### 2.2 Sampling and prompt-construction utilities

In [ ]:
def sample_pair(task_spec, label_pair, rng):
    """Sample one lexical pair belonging to a requested task class."""
    categories = task_spec["categories"]

    left_cat = categories[label_pair[0]]
    right_cat = categories[label_pair[1]]

    left = rng.choice(CATEGORIES[left_cat])
    right = rng.choice(CATEGORIES[right_cat])

    return {
        "left": left,
        "right": right,
        "left_category": left_cat,
        "right_category": right_cat,
        "label": task_spec["mapping"][label_pair],
        "class_pair": list(label_pair),
    }


def demo_labels(task_spec, shots, rng):
    """Choose demonstration classes, covering every class when shots allow it."""
    classes = list(task_spec["mapping"].keys())

    if shots == 0:
        return []

    selected = []

    # When enough demonstrations are available, guarantee at least one
    # example from every class.
    if shots >= len(classes):
        selected.extend(classes)

    while len(selected) < shots:
        selected.append(rng.choice(classes))

    rng.shuffle(selected)
    return selected[:shots]


def balanced_query_classes(task_spec, n_trials, seed):
    """Create a deterministic, approximately uniform query-class schedule."""
    classes = list(task_spec["mapping"].keys())
    rng = random.Random(seed)

    remainder_order = classes.copy()
    rng.shuffle(remainder_order)

    base = n_trials // len(classes)
    remainder = n_trials % len(classes)

    schedule = []
    for cls in classes:
        schedule.extend([cls] * base)

    schedule.extend(remainder_order[:remainder])
    rng.shuffle(schedule)

    assert len(schedule) == n_trials
    return schedule

In [ ]:
def build_icl_prompt(task_name, shots, seed, query_class=None):
    """Build one ICL prompt and return both the prompt and its metadata."""
    rng = random.Random(seed)
    task = TASKS[task_name]

    selected_labels = demo_labels(task, shots, rng)

    used_pairs = set()
    demos = []

    # Generate unique demonstration pairs.
    for label_pair in selected_labels:
        for _ in range(100):
            ex = sample_pair(task, label_pair, rng)
            key = (ex["left"], ex["right"])

            if key not in used_pairs:
                used_pairs.add(key)
                demos.append(ex)
                break

    # Query class can be fixed externally for exact class balancing.
    if query_class is None:
        query_class = rng.choice(list(task["mapping"].keys()))

    # Ensure the query lexical pair is not reused from the demonstrations.
    while True:
        query = sample_pair(task, query_class, rng)
        key = (query["left"], query["right"])

        if key not in used_pairs:
            break

    lines = [
        f'{ex["left"]}, {ex["right"]}: {ex["label"]}'
        for ex in demos
    ]

    # Keep trailing whitespace after the colon.
    lines.append(f'{query["left"]}, {query["right"]}: ')
    prompt = "\n".join(lines)

    return {
        "task": task_name,
        "shots": shots,
        "prompt": prompt,
        "gold_label": query["label"],
        "expected_continuation": query["label"],
        "allowed_labels": sorted(set(task["mapping"].values())),
        "query": query,
        "demos": demos,
        "seed": seed,
    }

### 2.3 Sanity-check one prompt

In [ ]:
example = build_icl_prompt(
    task_name="four_class",
    shots=5,
    seed=123,
)

print(example["prompt"])
print()
print("Gold:", example["gold_label"])

### 2.4 Generate the complete ICL dataset

The shot counts are:

\[
0,1,2,3,4,5,10,20
\]

We generate 100 queries per task × shot condition. Query classes are balanced deterministically within each condition.

In [ ]:
SHOT_COUNTS = [0, 1, 2, 3, 4, 5, 10, 20]
N_TRIALS = 100

icl_rows = []

for task_idx, task_name in enumerate(TASKS):
    task_spec = TASKS[task_name]

    for shots in SHOT_COUNTS:
        schedule_seed = (
            BASE_SEED
            + task_idx * 1_000_000
            + shots * 10_000
        )

        query_classes = balanced_query_classes(
            task_spec=task_spec,
            n_trials=N_TRIALS,
            seed=schedule_seed,
        )

        for trial in range(N_TRIALS):
            seed = (
                BASE_SEED
                + task_idx * 1_000_000
                + shots * 10_000
                + trial
            )

            row = build_icl_prompt(
                task_name=task_name,
                shots=shots,
                seed=seed,
                query_class=query_classes[trial],
            )

            row["id"] = f"{task_name}/shots{shots}/trial{trial:03d}"
            row["trial"] = trial
            row["stream"] = "icl_pattern_discovery"

            icl_rows.append(row)

print(f"Generated {len(icl_rows):,} ICL prompts.")

### 2.5 Validate class balance and save

In [ ]:
df_icl = pd.DataFrame([
    {
        "id": row["id"],
        "task": row["task"],
        "shots": row["shots"],
        "gold": row["gold_label"],
    }
    for row in icl_rows
])

print("Rows per task × shot condition:")
display(
    df_icl.groupby(["task", "shots"])
    .size()
    .unstack()
)

print("Gold-label counts per task × shot condition:")
display(
    df_icl.groupby(["task", "shots", "gold"])
    .size()
    .rename("count")
    .reset_index()
)

In [ ]:
icl_path = DATA_DIR / "icl_stream.jsonl"
write_jsonl(icl_rows, icl_path)

### 2.6 OLMo tokenizer audit

The later evaluation uses one-token label generation, so verify how OLMo tokenizes the numeric labels.

The continuation helper checks tokenization **in prompt context**, which is the relevant setting at generation time.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)


def continuation_token_ids(prompt, continuation):
    """Return token IDs introduced when `continuation` is appended to `prompt`."""
    base = tokenizer.encode(prompt, add_special_tokens=False)
    full = tokenizer.encode(prompt + continuation, add_special_tokens=False)
    return full[len(base):]

In [ ]:
audit_prompt = "apple, January: "

print("PROMPT IDS:", tokenizer.encode(audit_prompt, add_special_tokens=False))
print(
    "PROMPT TOKENS:",
    tokenizer.convert_ids_to_tokens(
        tokenizer.encode(audit_prompt, add_special_tokens=False)
    ),
)

print("\nLABEL CONTINUATIONS:")
for label in [str(i) for i in range(9)]:
    ids = continuation_token_ids(audit_prompt, label)
    print(label, ids, tokenizer.convert_ids_to_tokens(ids))

### 2.7 Metrics to compute later

For every checkpoint, task, shots

the evaluation stage can compute:

- **Format accuracy** — whether the model generated a valid label.
- **Task accuracy** — whether the generated valid label matches the gold label.

This notebook only constructs and audits the data; model evaluation belongs in a separate notebook/script.

# 3. Data Stream 2 — AGENDA Semantic-RI Examples

For the semantic induction-head analysis, this stream uses the AGENDA test set.

Pipeline:

1. Download the preprocessed AGENDA test data and relation vocabulary.
2. Parse graph triples and text.
3. Replace entities with safe single capital letters.
4. Split paragraphs into sentences.
5. Keep relation instances where both entities occur exactly once in the same sentence.
6. Audit head/tail spans with the OLMo tokenizer.
7. Save only token-safe examples for the final experiment.

### 3.1 Download and load AGENDA

In [ ]:
!wget -q \
  https://raw.githubusercontent.com/rikdz/GraphWriter/master/data/preprocessed.test.tsv \
  -O /content/preprocessed.test.tsv

!wget -q \
  https://raw.githubusercontent.com/rikdz/GraphWriter/master/data/relations.vocab \
  -O /content/relations.vocab

In [ ]:
AGENDA_PATH = Path("/content/preprocessed.test.tsv")
REL_PATH = Path("/content/relations.vocab")

relations = [
    line.strip()
    for line in REL_PATH.read_text().splitlines()
    if line.strip()
]

print(f"Loaded {len(relations)} relation labels.")
print(relations)

### 3.2 Parse AGENDA records

In [ ]:
def parse_agenda_line(line, sample_id):
    """Parse one preprocessed AGENDA TSV row."""
    parts = line.rstrip("\n").split("\t")

    if len(parts) < 5:
        return None

    title = parts[0]
    entities = [x.strip() for x in parts[1].split(";")]
    entity_types = parts[2].split()
    graph_text = parts[3]
    paragraph = parts[4]

    triples = []

    if graph_text.strip():
        for raw in graph_text.split(";"):
            values = raw.strip().split()

            if len(values) != 3:
                continue

            head_idx, relation_idx, tail_idx = map(int, values)

            triples.append({
                "head_idx": head_idx,
                "relation_idx": relation_idx,
                "relation": relations[relation_idx],
                "tail_idx": tail_idx,
            })

    return {
        "sample_id": sample_id,
        "title": title,
        "entities": entities,
        "entity_types": entity_types,
        "paragraph": paragraph,
        "triples": triples,
    }

In [ ]:
agenda_samples = []

with open(AGENDA_PATH, encoding="utf-8") as f:
    for sample_id, line in enumerate(f):
        sample = parse_agenda_line(line, sample_id)

        if sample is not None:
            agenda_samples.append(sample)

print(f"Parsed {len(agenda_samples):,} AGENDA samples.")

### 3.3 Replace entities with safe single-letter symbols

In [ ]:
# The experiment uses single capital letters so the relation head/tail
# can correspond cleanly to individual vocabulary tokens.
# Exclude letters that commonly have standalone linguistic meaning.
EXCLUDED_LETTERS = set("AINSWE")
SAFE_LETTERS = [
    c for c in string.ascii_uppercase
    if c not in EXCLUDED_LETTERS
]

ENTITY_PATTERN = re.compile(r"<[^>]+_(\d+)>")

print("Safe letters:", SAFE_LETTERS)
print("Count:", len(SAFE_LETTERS))

In [ ]:
def replace_entities_with_letters(sample):
    """Replace AGENDA entity placeholders with deterministic safe letters."""
    n_entities = len(sample["entities"])

    # A faithful one-letter representation is impossible if the sample
    # contains more entities than available symbols.
    if n_entities > len(SAFE_LETTERS):
        return None

    mapping = {
        entity_idx: SAFE_LETTERS[entity_idx]
        for entity_idx in range(n_entities)
    }

    def replace(match):
        idx = int(match.group(1))
        return mapping.get(idx, match.group(0))

    text = ENTITY_PATTERN.sub(replace, sample["paragraph"])
    return text, mapping

### 3.4 Sentence segmentation and relation-row construction

In [ ]:
nlp = spacy.load("en_core_web_sm")


def standalone_occurrences(text, symbol):
    """Find standalone occurrences of a capital-letter entity symbol."""
    return [
        match.span()
        for match in re.finditer(
            rf"(?<![A-Z]){re.escape(symbol)}(?![A-Z])",
            text,
        )
    ]


def build_agenda_relation_rows(samples):
    """Build sentence-level relation examples with unambiguous entity spans."""
    rows = []

    for sample in samples:
        replaced = replace_entities_with_letters(sample)

        if replaced is None:
            continue

        text, mapping = replaced
        doc = nlp(text)

        sentences = [
            sent.text.strip()
            for sent in doc.sents
            if sent.text.strip()
        ]

        for triple_idx, triple in enumerate(sample["triples"]):
            head_idx = triple["head_idx"]
            tail_idx = triple["tail_idx"]

            if head_idx not in mapping or tail_idx not in mapping:
                continue

            head_letter = mapping[head_idx]
            tail_letter = mapping[tail_idx]

            for sent_idx, sentence in enumerate(sentences):
                head_occ = standalone_occurrences(sentence, head_letter)
                tail_occ = standalone_occurrences(sentence, tail_letter)

                # Keep only unambiguous single occurrences.
                if len(head_occ) != 1 or len(tail_occ) != 1:
                    continue

                rows.append({
                    "id": (
                        f'agenda/{sample["sample_id"]:04d}/'
                        f'{triple["relation"]}/'
                        f'triple{triple_idx}/sent{sent_idx}'
                    ),
                    "stream": "agenda_semantic_ri",
                    "sample_id": sample["sample_id"],
                    "relation": triple["relation"],
                    "text": sentence,
                    "head_entity": sample["entities"][head_idx],
                    "tail_entity": sample["entities"][tail_idx],
                    "head_letter": head_letter,
                    "tail_letter": tail_letter,
                    "head_span": list(head_occ[0]),
                    "tail_span": list(tail_occ[0]),
                    "head_entity_index": head_idx,
                    "tail_entity_index": tail_idx,
                })

    return rows

### 3.5 Generate, inspect, and save the raw RI stream

In [ ]:
ri_rows = build_agenda_relation_rows(agenda_samples)

print(f"Generated {len(ri_rows):,} relation rows.")

df_ri = pd.DataFrame(ri_rows)

print("Relation coverage:")
display(
    df_ri["relation"]
    .value_counts()
    .rename_axis("relation")
    .reset_index(name="count")
)

In [ ]:
ri_path = DATA_DIR / "ri_agenda_stream.jsonl"
write_jsonl(ri_rows, ri_path)

### 3.6 OLMo token-safety audit

In [ ]:
def span_to_token_positions(offsets, span):
    """Return token positions overlapping a character span."""
    start, end = span

    return [
        i
        for i, (token_start, token_end) in enumerate(offsets)
        if token_start < end and token_end > start
    ]


def audit_ri_row(row):
    """Check whether head and tail entity symbols each map to one OLMo token."""
    enc = tokenizer(
        row["text"],
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    offsets = enc["offset_mapping"]

    head_positions = span_to_token_positions(
        offsets,
        row["head_span"],
    )
    tail_positions = span_to_token_positions(
        offsets,
        row["tail_span"],
    )

    result = dict(row)
    result["head_token_positions"] = head_positions
    result["tail_token_positions"] = tail_positions

    result["token_safe"] = (
        len(head_positions) == 1
        and len(tail_positions) == 1
    )

    if result["token_safe"]:
        result["head_token_id"] = enc["input_ids"][head_positions[0]]
        result["tail_token_id"] = enc["input_ids"][tail_positions[0]]

    return result

In [ ]:
audited_ri = [audit_ri_row(row) for row in ri_rows]
safe_ri = [row for row in audited_ri if row["token_safe"]]

print("All rows:", len(audited_ri))
print("OLMo-token-safe rows:", len(safe_ri))

if audited_ri:
    print("Safe fraction:", len(safe_ri) / len(audited_ri))

In [ ]:
safe_ri_path = DATA_DIR / "ri_agenda_olmo_safe.jsonl"
write_jsonl(safe_ri, safe_ri_path)

## 4. Optional Sensitivity Analysis — Remove spaCy Stopwords

Ren et al. state that frequent function words annotated by spaCy are removed, but the exact list is not specified. Therefore, this should be treated as a **sensitivity-analysis variant**, not as an exact reconstruction.

In [ ]:
def remove_spacy_stopwords_preserve_letters(text):
    """Remove spaCy stopwords while preserving entity letters and punctuation."""
    doc = nlp(text)
    kept = []

    for tok in doc:
        if (
            tok.text in SAFE_LETTERS
            or not tok.is_stop
            or tok.is_punct
        ):
            kept.append(tok.text)

    return " ".join(kept)

# 5. Final Dataset Summary

This cell provides a compact end-of-notebook check after all generation steps have run.

In [ ]:
summary = pd.DataFrame([
    {
        "dataset": "ICL pattern discovery",
        "rows": len(icl_rows),
        "path": str(icl_path),
    },
    {
        "dataset": "AGENDA raw RI",
        "rows": len(ri_rows),
        "path": str(ri_path),
    },
    {
        "dataset": "AGENDA OLMo-token-safe RI",
        "rows": len(safe_ri),
        "path": str(safe_ri_path),
    },
])

display(summary)